In [0]:
# Storage account access key set karo (authentication ke liye)
storage_account = "sonudedatalake01"
storage_account_key = "YOUR_KEY_HERE"

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.dfs.core.windows.net",
    storage_account_key
)

# Bronze layer se raw CSV read karo
df_bronze = spark.read.csv(
    f"abfss://bronze@{storage_account}.dfs.core.windows.net/sales_data.csv",
    header=True,
    inferSchema=True
)

display(df_bronze)

total_bill,tip,sex,smoker,day,time,size
16.99,1.01,Female,No,Sun,Dinner,2
10.34,1.66,Male,No,Sun,Dinner,3
21.01,3.5,Male,No,Sun,Dinner,3
23.68,3.31,Male,No,Sun,Dinner,2
24.59,3.61,Female,No,Sun,Dinner,4
25.29,4.71,Male,No,Sun,Dinner,4
8.77,2.0,Male,No,Sun,Dinner,2
26.88,3.12,Male,No,Sun,Dinner,4
15.04,1.96,Male,No,Sun,Dinner,2
14.78,3.23,Male,No,Sun,Dinner,2


In [0]:
# Silver layer: duplicates aur null values clean karo
df_silver = df_bronze.dropDuplicates().na.drop()

# Silver container mein Delta format mein save karo
df_silver.write.format("delta").mode("overwrite").save(
    f"abfss://silver@{storage_account}.dfs.core.windows.net/sales_silver"
)

display(df_silver)

total_bill,tip,sex,smoker,day,time,size
34.3,6.7,Male,No,Thur,Lunch,6
20.92,4.08,Female,No,Sat,Dinner,2
16.93,3.07,Female,No,Sat,Dinner,3
24.55,2.0,Male,No,Sun,Dinner,4
17.81,2.34,Male,No,Sat,Dinner,4
13.81,2.0,Male,Yes,Sat,Dinner,2
32.68,5.0,Male,Yes,Thur,Lunch,2
22.82,2.18,Male,No,Thur,Lunch,3
30.06,2.0,Male,Yes,Sat,Dinner,3
20.76,2.24,Male,No,Sun,Dinner,2


In [0]:
# Gold layer: business-level aggregation (day-wise total sales)
df_gold = df_silver.groupBy("day").sum("total_bill", "tip")

df_gold = df_gold.withColumnRenamed("sum(total_bill)", "total_sales") \
                  .withColumnRenamed("sum(tip)", "total_tips")

# Gold container mein Delta format mein save karo
df_gold.write.format("delta").mode("overwrite").save(
    f"abfss://gold@{storage_account}.dfs.core.windows.net/sales_gold"
)

display(df_gold)

day,total_sales,total_tips
Thur,1083.3299999999995,169.83000000000004
Sun,1627.1600000000003,247.38999999999996
Sat,1778.3999999999996,260.4
Fri,325.88000000000005,51.96
